# 01 · Descarga de datos StatsBomb — Club América (Liga MX)

Descarga **todos** los partidos del América disponibles en la API (2021/22 → 2026/27, todos los entrenadores) y guarda por partido:

| dataset | contenido |
|---|---|
| `events` | eventos (pases, conducciones, tiros, presiones…) con xG, OBV, `is_transition`, `is_controlled_possession` |
| `frames360` | posiciones de jugadores visibles en cada evento (StatsBomb 360) |
| `lineups` | alineaciones, posiciones y minutos por jugador |
| `player_match_stats` | métricas agregadas por jugador y partido |
| `team_match_stats` | métricas agregadas por equipo y partido (PPDA, xG, posesión…) |

Es **reanudable**: lo que ya existe en `data/raw/` se salta. Vuelve a correrlo durante la temporada 2026/27 para sumar partidos nuevos de Almada.

> Requiere `.env` con `SB_USERNAME` / `SB_PASSWORD` (ver `.env.example`). Los datos son de uso exclusivo de la competencia: `data/` está en `.gitignore`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from statsbombpy import sb
from tqdm.auto import tqdm

from src.sb_client import (DATASETS, LIGA_MX, PROCESSED, RAW, SEASONS_LIGA_MX,
                           download_match, get_creds, get_team_matches,
                           load_dataset, to_parquet_safe)

TEAM = "América"
N_WORKERS = 4  # descargas en paralelo; bajar si la API responde con errores 429
creds = get_creds()

## 1. Competencias y temporadas disponibles con nuestras credenciales

In [2]:
comps = sb.competitions(creds=creds)
comps[["competition_id", "competition_name", "season_id", "season_name", "match_available", "match_available_360"]]

,competition_id,competition_name,season_id,season_name,match_available,match_available_360
0,73,Liga MX,351,2026/2027,2026-09-29T16:39:29.000000,2026-09-29T16:39:29.000000
1,73,Liga MX,318,2025/2026,2026-09-29T16:52:08.000000,2026-09-29T16:52:08.000000
2,73,Liga MX,317,2024/2025,2026-07-01T20:17:42.000000,2026-07-01T20:17:42.000000
3,73,Liga MX,281,2023/2024,2026-09-29T08:04:23.000000,2026-09-29T08:04:23.000000
4,73,Liga MX,235,2022/2023,2026-09-29T07:38:23.000000,2026-09-29T07:38:23.000000
5,73,Liga MX,108,2021/2022,2026-03-13T13:02:37.000000,2026-03-13T13:02:37.000000
6,1438,Liga MX Femenil,351,2026/2027,2026-09-29T16:08:26.000000,NaN
7,1438,Liga MX Femenil,318,2025/2026,2026-09-11T23:36:11.000000,NaN
8,1438,Liga MX Femenil,317,2024/2025,2025-06-16T18:18:19.434166,NaN


## 2. Partidos del América (con entrenador, rival, localía y resultado)

In [3]:
matches = get_team_matches(TEAM, creds, competition_id=LIGA_MX, seasons=SEASONS_LIGA_MX)
to_parquet_safe(matches, RAW / "matches_america.parquet")

print(matches.match_status.value_counts().to_dict())
(matches[matches.match_status == "available"]
 .groupby(["season", "team_manager"])
 .agg(partidos=("match_id", "size"), desde=("match_date", "min"), hasta=("match_date", "max"),
      G=("result", lambda r: (r == "W").sum()), E=("result", lambda r: (r == "D").sum()),
      P=("result", lambda r: (r == "L").sum()),
      GF=("goals_for", "sum"), GC=("goals_against", "sum")))

{'available': 222, 'scheduled': 8}


partidos       desde       hasta  \
season    team_manager                                                       
2021/2022 Fernando Ortiz                        13  2022-03-06  2022-05-23   
          Santiago Hernán Solari Poggio         27  2021-07-23  2022-03-02   
2022/2023 Fernando Ortiz                        42  2022-07-03  2023-05-22   
2023/2024 André Soares Jardine                  46  2023-07-01  2024-05-27   
2024/2025 André Soares Jardine                  45  2024-07-06  2025-05-26   
          Diego Alberto Cervantes Chávez         2  2025-01-11  2025-01-17   
2025/2026 André Soares Jardine                  38  2025-07-12  2026-05-11   
2026/2027 Jorge Guillermo Almada Álves           9  2026-07-19  2026-09-28   

                                           G   E   P    GF    GC  
season    team_manager                                            
2021/2022 Fernando Ortiz                   7   4   2  20.0  10.0  
          Santiago Hernán Solari Poggio   11   9   7  31.0  27.0  
2022/2023 Fernando Ortiz                  25  10   7  93.0  49.0  
2023/2024 André Soares Jardine            27  15   4  85.0  34.0  
2024/2025 André Soares Jardine            23  11  11  76.0  41.0  
          Diego Alberto Cervantes Chávez   1   1   0   2.0   1.0  
2025/2026 André Soares Jardine            18  10  10  61.0  44.0  
2026/2027 Jorge Guillermo Almada Álves     6   2   1  21.0  10.0

## 3. Descarga por partido (reanudable, en paralelo)

In [4]:
played = matches[matches.match_status == "available"]
match_ids = played.match_id.astype(int).tolist()

log = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(download_match, mid, creds): mid for mid in match_ids}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="partidos"):
        log.append({"match_id": futures[fut], **fut.result()})

log = pd.DataFrame(log).sort_values("match_id")
log.to_csv(RAW / "download_log.csv", index=False)
log.drop(columns="match_id").apply(lambda s: s.str.split(":").str[0].value_counts()).fillna(0).astype(int)

partidos:   0%|          | 0/222 [00:00<?, ?it/s]

https://data.statsbombservices.com/api/v4/matches/3889728/team-stats -> 502


https://data.statsbombservices.com/api/v2/360-frames/3917452 -> 502


,events,frames360,lineups,player_match_stats,team_match_stats
cached,1,1,1,1,1
error,0,2,0,0,0
ok,221,219,221,221,221


### Reintentar los que fallaron (si hay)

In [5]:
failed = log[log.drop(columns="match_id").apply(lambda s: s.str.startswith("error")).any(axis=1)]
print(f"{len(failed)} partidos con algún error")
for mid in tqdm(failed.match_id, desc="reintentos"):
    print(mid, download_match(int(mid), creds))

2 partidos con algún error


reintentos:   0%|          | 0/2 [00:00<?, ?it/s]

3917452 {'events': 'cached', 'frames360': 'ok', 'lineups': 'cached', 'player_match_stats': 'cached', 'team_match_stats': 'cached'}
4038856 {'events': 'cached', 'frames360': "error: KeyError: 'freeze_frame'", 'lineups': 'cached', 'player_match_stats': 'cached', 'team_match_stats': 'cached'}


## 4. Consolidar en `data/processed/`

Se agregan columnas de contexto del partido (temporada, entrenador, rival, localía) para poder cortar todo por entrenador y contexto. Los `frames360` se dejan por partido en `data/raw/frames360/` (son pesados); se cargan con `load_dataset("frames360", match_ids)` cuando se necesiten.

In [6]:
ctx = played[["match_id", "match_date", "season", "match_week", "team_manager", "opponent",
               "is_home", "goals_for", "goals_against", "result"]]

for name in ["events", "lineups", "player_match_stats", "team_match_stats"]:
    df = load_dataset(name, match_ids).merge(ctx, on="match_id", how="left")
    to_parquet_safe(df, PROCESSED / f"{name}_america.parquet")
    print(f"{name:20s} {df.shape}")

events               (736829, 162)


lineups              (9317, 25)


player_match_stats   (6870, 187)


team_match_stats     (442, 216)


## 5. Chequeos rápidos

In [7]:
events = pd.read_parquet(PROCESSED / "events_america.parquet")
n360 = len(list((RAW / "frames360").glob("*.parquet")))
print(f"Partidos con eventos: {events.match_id.nunique()} / {len(match_ids)} | con 360: {n360}")

ev_am = events[events.team == TEAM]
(ev_am.groupby(["season", "team_manager"])
 .agg(partidos=("match_id", "nunique"), eventos=("id", "size"),
      pases=("type", lambda t: (t == "Pass").sum()),
      tiros=("type", lambda t: (t == "Shot").sum()),
      xG=("shot_statsbomb_xg", "sum")))

Partidos con eventos: 222 / 222 | con 360: 221


partidos  eventos  pases  tiros  \
season    team_manager                                                      
2021/2022 Fernando Ortiz                        13    19612   5355    186   
          Santiago Hernán Solari Poggio         27    41227  11443    342   
2022/2023 Fernando Ortiz                        42    76205  21680    700   
2023/2024 André Soares Jardine                  46    88137  25392    622   
2024/2025 André Soares Jardine                  45    80988  23009    595   
          Diego Alberto Cervantes Chávez         2     3292    916     41   
2025/2026 André Soares Jardine                  38    68382  20034    563   
2026/2027 Jorge Guillermo Almada Álves           9    15617   4617    182   

                                                 xG  
season    team_manager                               
2021/2022 Fernando Ortiz                  22.407731  
          Santiago Hernán Solari Poggio   31.691306  
2022/2023 Fernando Ortiz                  70.939832  
2023/2024 André Soares Jardine            83.705832  
2024/2025 André Soares Jardine            69.829422  
          Diego Alberto Cervantes Chávez   4.796676  
2025/2026 André Soares Jardine            52.676812  
2026/2027 Jorge Guillermo Almada Álves    17.266089

In [8]:
# Tipos de evento del América (para ubicarse en el diccionario de StatsBomb)
ev_am.type.value_counts()

type
Pass                 112446
Ball Receipt*        104047
Carry                 92204
Pressure              31506
Ball Recovery         10188
Duel                   6749
Clearance              3931
Block                  3717
Shot                   3231
Dribble                3188
Foul Won               3058
Miscontrol             3030
Goal Keeper            2932
Dispossessed           2759
Foul Committed         2551
Interception           1855
Dribbled Past          1501
Substitution           1016
Injury Stoppage         677
Half Start              447
Half End                447
50/50                   434
Referee Ball-Drop       377
Tactical Shift          320
Starting XI             222
Shield                  151
Player Off              137
Player On               136
Bad Behaviour           113
Offside                  42
Error                    36
Own Goal For              7
Own Goal Against          5
Name: count, dtype: int64